In [29]:
import requests
import re
from Bio import SeqIO
import subprocess


class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
        response = requests.get(url, headers={"Accept": "application/json"})
        if response.status_code != 200:
            return {"error": "Request failed"}
        return response.json()

    def _get_ensembl(self, gene_id):
        url = f"https://rest.ensembl.org/lookup/id/{gene_id}"
        response = requests.get(url, headers={"Accept": "application/json"})
        if response.status_code != 200:
            return {"error": "Request failed"}
        return response.json()

    def _uniprot_parse_response(self, resp: dict):
        if "error" in resp:
            return resp
        try:
            return {
                "organism": resp.get("organism", {}).get("scientificName", "NA"),
                "geneInfo": resp.get("genes", []),
                "sequenceInfo": resp.get("sequence", {}),
                "type": resp.get("uniProtkbEntryType", "protein"),
            }
        except Exception as e:
            return {"error": f"Parse failed: {str(e)}"}

    def _ensembl_parse_response(self, resp: dict):
        if "error" in resp:
            return resp
        try:
            if "error" in resp:
                return {"error": resp["error"]}
            return {
                "object_type": resp.get("object_type", "NA"),
                "species": resp.get("species", "NA"),
                "assembly_name": resp.get("assembly_name", "NA"),
                "biotype": resp.get("biotype", "NA"),
                "display_name": resp.get("display_name", "NA"),
                "id": resp.get("id", "NA"),
                "description": resp.get("description", "NA"),
            }
        except Exception as e:
            return {"error": f"Parse failed: {str(e)}"}

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        if not id:
            return {"error": "No ID match found."}

        if database == "uniprot":
            resp = self._get_uniprot(id)
            return self._uniprot_parse_response(resp)

        elif database == "ensembl":
            resp = self._get_ensembl(id)
            return self._ensembl_parse_response(resp)

        return {"error": "Unknown database"}

    def seqkit_stats(self) -> dict:
        try:
            result = subprocess.run(
                ["seqkit", "stats", self.filename],
                capture_output=True,
                text=True,
                check=True,
            )

            lines = result.stdout.strip().split("\n")

            if len(lines) < 2:
                return {"error": "Invalid seqkit output"}

            headers = lines[0].split()[1:]
            values = lines[1].split()[1:]

            stats = dict(zip(headers, values))

            return {
                "fasta_seqkit_stat_info": stats,
                "fasta_type": stats.get("type", "Protein"),
                "fasta_num_seqs": int(stats.get("num_seqs", 0)),
            }

        except subprocess.CalledProcessError as e:
            return {"error": e.stderr.strip()}

    def biopython_parser(self, seqkit_result) -> dict:
        if "error" in seqkit_result:
            return seqkit_result

        fasta_type = seqkit_result["fasta_type"]

        if fasta_type == "Protein":
            id_pattern = re.compile(r"(?:sp\|)?([OPQ][0-9][A-Z0-9]{3}[0-9])")
            database = "uniprot"
        elif fasta_type == "DNA":
            id_pattern = re.compile(r"(ENS[A-Z]*\d+)")
            database = "ensembl"
        else:
            return {"error": f"Unknown type: {fasta_type}"}

        results = []

        try:
            with open(self.filename) as handle:
                records = list(SeqIO.parse(handle, "fasta"))

                if not records:
                    return {"error": "Invalid FASTA file"}

                for record in records:
                    desc = record.description
                    seq = str(record.seq)

                    match = id_pattern.search(desc)
                    db_id = match.group(1) if match else None

                    db_info = self._access_database(
                        db_id, database, desc, seq
                    )

                    results.append({
                        f"file_info_{db_id or 'unknown'}": {
                            "description": desc,
                            "sequence": seq,
                            f"database_info_{db_id or 'unknown'}": db_info,
                        }
                    })

        except Exception as e:
            return {"error": f"FASTA parsing failed: {str(e)}"}

        return {
            "DB_name": database,
            "results": results,
            "stats": seqkit_result,
        }

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            elif isinstance(value, list):
                for item in value:
                    self.show_output(item, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [30]:
parser = MyFastaParser("test_file.fasta")

stats = parser.seqkit_stats()
print(stats)

biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228', 'max_len': '427'}, 'fasta_type': 'Protein', 'fasta_num_seqs': 2}
DB_name
	uniprot
results
	file_info_P11473
		description
			sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
		sequence
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
		database_info_P11473
			organism
				Homo sapiens
			geneInfo
				geneName
					evidences
						evidenceCode
							ECO:0000312
						source
							HGNC
						id
							HGNC:12679
					value
						VDR
				synonyms


In [31]:
parser = MyFastaParser("ensembl_download_1.fasta")

stats = parser.seqkit_stats()
print(stats)

biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'fasta_seqkit_stat_info': {'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23'}, 'fasta_type': 'DNA', 'fasta_num_seqs': 6}
DB_name
	ensembl
results
	file_info_ENSMUST00000196221
		description
			ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
		sequence
			ATGGCATAT
		database_info_ENSMUST00000196221
			object_type
				Transcript
			species
				mus_musculus
			assembly_name
				GRCm39
			biotype
				TR_D_gene
			display_name
				Trdd1-202
			id
				ENSMUST00000196221
			description
				NA
	file_info_ENSMUST00000177564
		description
			ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta

In [33]:
parser = MyFastaParser("ensembl_download_2.fasta")

stats = parser.seqkit_stats()
print(stats)

biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'error': 'Invalid seqkit output'}
error
	Invalid seqkit output
